# Chapitre 2 — PyTorch de base

Tenseurs, autograd, et un petit réseau de neurones de **vision**.

Ce chapitre **tourne sans GPU** (CPU OK) et sans données RSNA : on apprend sur le
jeu jouet `digits` de scikit-learn (chiffres 8×8), déjà embarqué, aucun
téléchargement.

## À propos des organigrammes (flowcharts)

Un organigramme écrit en **Mermaid** (un bloc ```` ```mermaid ```` dans une cellule
Markdown) ne s'affiche que dans JupyterLab ≥ 4.1 avec le rendu Mermaid activé —
sinon la cellule reste **blanche**. C'est pour ça que tes flowcharts n'étaient pas
visibles.

La solution robuste : on **génère le diagramme depuis une cellule de code**, dont la
sortie image est sauvegardée dans le `.ipynb`. Ça marche dans tout Jupyter, dans
`nbconvert` et dans l'aperçu GitHub. On utilise **matplotlib** (déjà installé). Le
helper `flowchart()` ci-dessous est réutilisé dans les chapitres suivants.

In [ ]:
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('CUDA dispo :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def flowchart(steps, title=None, width=8.5, box_h=0.62, gap=0.45,
              facecolor='#e7f0fb', edgecolor='#2b6cb0', fontsize=11):
    '''Organigramme vertical : une boîte par étape, flèches entre elles.
    Rendu en SORTIE d une cellule de code -> visible partout (contrairement au Mermaid).'''
    n = len(steps)
    unit = box_h + gap
    fig, ax = plt.subplots(figsize=(width, n * unit + 0.3))
    ax.set_xlim(0, 10)
    ax.set_ylim(-gap, n * unit)
    ax.axis('off')
    for i, label in enumerate(steps):
        y = (n - 1 - i) * unit
        ax.add_patch(FancyBboxPatch((1, y), 8, box_h,
                     boxstyle='round,pad=0.08', linewidth=1.6,
                     facecolor=facecolor, edgecolor=edgecolor))
        ax.text(5, y + box_h / 2, label, ha='center', va='center', fontsize=fontsize)
        if i < n - 1:
            ax.annotate('', xy=(5, y - gap), xytext=(5, y),
                        arrowprops=dict(arrowstyle='-|>', color=edgecolor, lw=1.8))
    if title:
        ax.set_title(title, fontsize=fontsize + 2, fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()

# Démo : l organigramme d une boucle d entraînement PyTorch
flowchart([
    'Charger un batch (images, labels)',
    'forward : sorties = model(images)',
    'Calculer la loss',
    'loss.backward()  -> gradients (autograd)',
    'optimizer.step()  -> met a jour les poids',
    'optimizer.zero_grad()  -> remet les gradients a zero',
], title='Boucle entrainement (1 iteration)')

## 1. Tenseurs

Le tenseur est la structure de base de PyTorch : un tableau n-dimensionnel
(comme un `ndarray` numpy) qui sait **vivre sur GPU** et **porter un graphe de
gradients**. Une image en niveaux de gris est un tenseur `(H, W)` ; un batch
d'images est `(N, C, H, W)`.

In [ ]:
x = torch.arange(6.).reshape(2, 3)
print('x =\n', x)
print('shape :', tuple(x.shape), '| dtype :', x.dtype)
print('somme :', x.sum().item(), '| moyenne :', x.mean().item())
print('x.T =\n', x.T)

# Vers/depuis numpy (même mémoire) et transfert GPU si dispo
import numpy as np
a = np.array([[1., 2.], [3., 4.]])
t = torch.from_numpy(a)
print('depuis numpy :', t.tolist())
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('tenseur sur', t.to(dev).device)

## 2. Autograd

Quand un tenseur a `requires_grad=True`, PyTorch enregistre les opérations dans un
**graphe**. `loss.backward()` remonte ce graphe et remplit `.grad` pour chaque
paramètre. C'est exactement ce qui se passe à l'étape `backward()` de
l'organigramme ci-dessus.

Vérifions sur une fonction où l'on connaît la dérivée : `y = x^2` donne
`dy/dx = 2x`.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()                 # calcule dy/dx
print('x =', x.item(), '| y = x^2 =', y.item())
print('dy/dx =', x.grad.item(), '(attendu : 2*x =', 2 * x.item(), ')')

## 3. Un petit réseau de vision

On définit un mini-CNN qui classe des chiffres 8×8 (`sklearn.datasets.load_digits`,
10 classes). Même ossature qu'un classifieur de mammographies, en miniature :
`Conv2d → ReLU → pooling → couche linéaire`. Ici les sorties sont des **logits**
(une par classe) → on utilise `CrossEntropyLoss` (qui applique le softmax en
interne). ⚠️ Au chapitre 5, GMIC sortira déjà des **probabilités** → on y utilisera
`binary_cross_entropy`, surtout pas `BCEWithLogits` (double sigmoïde).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
digits = load_digits()
X = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0  # (N,1,8,8) dans [0,1]
y = torch.tensor(digits.target, dtype=torch.long)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print('train', tuple(Xtr.shape), '| test', tuple(Xte.shape))

class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.fc = nn.Linear(16 * 2 * 2, n_classes)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)   # 8x8 -> 4x4
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)   # 4x4 -> 2x2
        return self.fc(x.flatten(1))                 # logits (N, 10)

model = SmallCNN()
print(model)
print('params :', sum(p.numel() for p in model.parameters()))

## 4. La boucle d'entraînement

On exécute concrètement les 6 étapes de l'organigramme du début. C'est le squelette
commun à **tous** les entraînements du cours (ch3 ResNet, ch5 GMIC).

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(8):
    model.train()
    for images, labels in train_loader:          # 1. batch
        logits = model(images)                    # 2. forward
        loss = loss_fn(logits, labels)            # 3. loss
        opt.zero_grad()                           # 6. (avant backward)
        loss.backward()                           # 4. gradients
        opt.step()                                # 5. mise a jour
    model.eval()
    with torch.no_grad():
        acc = (model(Xte).argmax(1) == yte).float().mean().item()
    print(f'epoch {epoch}  loss={loss.item():.3f}  test_acc={acc:.3f}')

## Récap

- **Tenseur** = tableau n-D, GPU-ready, qui porte le graphe de gradients.
- **Autograd** : `backward()` remplit `.grad` automatiquement.
- **CNN** : `Conv2d → ReLU → pool → Linear`.
- **Boucle** : batch → forward → loss → backward → step → zero_grad.
- **Flowcharts** : générés en code (`flowchart()`), jamais en Mermaid — ainsi ils
  s'affichent toujours.

Chapitre suivant → `03_resnet18_breast_density.ipynb` : un vrai réseau (ResNet-18)
sur de vraies mammographies.